# Part 1 — Getting Started

This notebook covers the fundamentals: installing the library, opening a connection, running raw SQL, and fetching data into a DataFrame.

**Notebooks in this series**

| # | File | Topic |
|---|------|-------|
| 1 | `Part1_Getting_Started.ipynb` | ← you are here |
| 2 | `Part2_Upsert_and_Schema.ipynb` | Upsert, conflict strategies, schema evolution |
| 3 | `Part3_Bulk_Operations.ipynb` | replace_table, delete_and_insert, COPY |
| 4 | `Part4_Edge_Cases.ipynb` | NaN, numpy types, column casing, empty frames … |
| 5 | `Part5_Advanced_psycopg3.ipynb` | Streaming, chunking, pgvector, pool tuning |

## 1. Installation

```bash
pip install pyposconnector
# psycopg 3 + SQLAlchemy 2 are pulled in automatically
```

If you want pgvector support (Part 5), also install:

```bash
pip install pgvector
```

## 2. Connecting to PostgreSQL

`PostgresConnector` wraps a SQLAlchemy engine backed by **psycopg 3**.  
The engine uses a connection pool, so you create it once and reuse it throughout your application.

In [ ]:
from postgres_connector import PostgresConnector

pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    port=5432,           # default — can be omitted
    schema="public",     # default — can be omitted
)

print("Connected!")

### Using a custom schema

Pass `schema` to isolate your tables from `public`.  
Every operation (DDL and DML) automatically targets that schema — you never have to qualify table names yourself.

In [ ]:
pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    schema="analytics",   # all tables live in the 'analytics' schema
)

# The schema is created automatically by PostgreSQL's search_path;
# you may need to create it first if it doesn't exist:
pg.execute_query("CREATE SCHEMA IF NOT EXISTS analytics;")
print("Schema ready.")

### Tuning the connection pool (optional)

The three most useful parameters for production use:

In [ ]:
pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    pool_size=10,        # persistent connections kept open (default 5)
    max_overflow=20,     # extra connections allowed under peak load (default 10)
    pool_timeout=30.0,   # seconds to wait before raising OperationalError (default 30)
)

## 3. Checking Whether a Table Exists

Use `check_table_exists()` before deciding whether to create or append.

In [ ]:
exists = pg.check_table_exists("orders")
print(f"Table 'orders' exists: {exists}")

# Practical pattern — avoid accidental overwrites:
if not pg.check_table_exists("orders"):
    print("Table not found — will be created on first upsert/replace.")
else:
    print("Table found — safe to append or upsert.")

## 4. Running Raw SQL with `execute_query()`

`execute_query()` runs any SQL statement (DDL or DML) inside a single transaction that auto-commits on success and rolls back on error.

### 4a. DDL — creating objects

In [ ]:
# Create a table manually (usually you let upsert_data do this, but it's possible)
pg.execute_query("""
    CREATE TABLE IF NOT EXISTS logs (
        id        BIGSERIAL PRIMARY KEY,
        event     TEXT NOT NULL,
        logged_at TIMESTAMP DEFAULT now()
    )
""")
print("Table created.")

### 4b. DML — parameterized INSERT / UPDATE / DELETE

Always use **named parameters** (`:name`) instead of f-strings to prevent SQL injection.

In [ ]:
# Safe parameterized INSERT
pg.execute_query(
    "INSERT INTO logs (event) VALUES (:evt)",
    params={"evt": "application started"},
)

# Safe parameterized UPDATE
pg.execute_query(
    "UPDATE logs SET event = :new_evt WHERE event = :old_evt",
    params={"new_evt": "app started", "old_evt": "application started"},
)

print("DML executed safely.")

### 4c. DROP

`execute_query` is also the right tool for cleanup — dropping tables, truncating, or removing indexes.

In [ ]:
pg.execute_query("DROP TABLE IF EXISTS logs;")
print("Cleaned up.")

## 5. Fetching Data with `get_data()`

`get_data()` executes a SELECT and returns a `pd.DataFrame`.  
It accepts the same named-parameter syntax as `execute_query()`.

### 5a. Basic SELECT

In [ ]:
df = pg.get_data("SELECT * FROM orders LIMIT 10;")
print(df.shape)   # (rows, cols)
df.head()

### 5b. Parameterized SELECT

In [ ]:
df = pg.get_data(
    "SELECT * FROM orders WHERE status = :s AND total > :min_total",
    params={"s": "shipped", "min_total": 100},
)
df.head()

### 5c. Streaming large result sets

By default, `get_data()` loads all rows into memory at once.  
For large queries, pass `stream=True` to use a **server-side cursor** — rows are fetched in small batches, keeping memory flat.

In [ ]:
# Same API — just add stream=True
df = pg.get_data(
    "SELECT * FROM orders",
    stream=True,          # uses a server-side cursor
    stream_buffer=5_000,  # rows buffered per fetch (default 2 000)
)
print(df.shape)

### 5d. Chunk iterator for very large tables

When you need to process rows incrementally without ever holding the full result in memory, use `get_data_chunks()` — it yields one `pd.DataFrame` per chunk.

In [ ]:
total_rows = 0
for chunk in pg.get_data_chunks("SELECT * FROM orders", chunksize=10_000):
    total_rows += len(chunk)
    # process chunk here — e.g. write to parquet, run transformations…

print(f"Processed {total_rows} rows total.")

## 6. Closing the Connection

Call `dispose()` when you are done.  
This gracefully closes all idle connections in the pool.

In [ ]:
pg.dispose()
print("Connection pool closed.")

## Summary

| Method | Purpose |
|--------|---------|
| `PostgresConnector(...)` | Create engine + pool |
| `check_table_exists(table)` | Boolean existence check |
| `execute_query(sql, params)` | DDL / DML, auto-commit |
| `get_data(sql, params)` | SELECT → DataFrame |
| `get_data(sql, stream=True)` | SELECT → DataFrame (server-side cursor) |
| `get_data_chunks(sql, chunksize)` | SELECT → iterator of DataFrames |
| `dispose()` | Close pool |

**Next:** [Part 2 — Upsert & Schema Evolution](Part2_Upsert_and_Schema.ipynb)